In [47]:
import random
import numpy as np
import os
import torch 

def set_seed(seed=24):
    """Setea semilla para reproducibilidad general"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # si usas multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Semilla fijada en: {seed}")

# Llamar a la función
set_seed(24)

Semilla fijada en: 24


In [48]:
import sys
sys.path.append('../')
 
import pandas as pd 
from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score
from plotly import express as px
from tutoriales.utils import plot_confusion_matrix, get_artifact_filename
from json import loads
from joblib import load, dump
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact
from optuna.visualization import plot_param_importances
from optuna.importance import FanovaImportanceEvaluator
from optuna.visualization import plot_contour

In [49]:
import sys, numpy.core
# Shim: permite cargar joblib guardados con numpy >= 2.0 en entornos con numpy 1.x
sys.modules.setdefault("numpy._core", numpy.core)
for _sub in ["numeric", "multiarray", "umath", "fromnumeric", "arrayprint", "strings"]:
    mod = getattr(numpy.core, _sub, numpy.core)
    sys.modules.setdefault(f"numpy._core.{_sub}", mod)

In [50]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

In [51]:
# Las predicciones del modelo LGB se guardan directamente como joblib
lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_1__variables_originales.joblib'))
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_2__variables_completas.joblib'))


In [52]:
MODEL_NAME = '01 DistilBert'
MODEL_VERSION = '5.0'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3", 
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)


[I 2026-05-09 00:38:02,518] Using an existing study with name '01 DistilBert_5.0' instead of creating a new one.


In [53]:
bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

In [54]:
# Cargar el modelo ResNet
MODEL_NAME_RESNET = '04 ResNet Augment'
MODEL_VERSION_RESNET = '1.0.0'

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/optuna_artifacts/db.sqlite3",
    study_name=f'{MODEL_NAME_RESNET}_{MODEL_VERSION_RESNET}',
    load_if_exists=True
)

[I 2026-05-09 00:38:02,610] Using an existing study with name '04 ResNet Augment_1.0.0' instead of creating a new one.


In [55]:
resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_04 ResNet Augment_1.0.0_1.joblib'))
#resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES,get_artifact_filename(study_resnet,'test')))

In [56]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

In [57]:
# Unir ResNet al dataframe fusionado
merged_datasets = merged_datasets.merge(
    resnet_dataset[['PetID', 'pred']].rename({'pred': 'resnet_pred_score'}, axis=1),
    on='PetID', how='outer'
)

In [58]:
merged_datasets.head()

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score,resnet_pred_score
0,002230dea,"[0.06175187846442381, 0.2940676868352113, 0.41...",1,"[0.009752763, 0.5465382, 0.43560302, 0.0048021...","[-0.99180895, 0.7428905, 0.48204118, 0.1142346..."
1,0063f83c9,"[0.08883536737434454, 0.16060991232554725, 0.2...",1,"[0.0020303065, 0.00568232, 0.088742584, 0.0133...","[-1.3278729, 0.98171896, -0.028774094, 0.35766..."
2,0073c33d0,"[0.006477767488363635, 0.222578069511566, 0.22...",3,"[0.0007240717, 0.016772836, 0.22749364, 0.7542...","[-1.4208313, 0.7716534, 1.2558718, 0.40233982,..."
3,00bfa5da9,"[0.008940191888288453, 0.09638043680171315, 0....",4,"[8.41588e-05, 0.00040059086, 0.0027940436, 0.0...","[-2.9708593, -0.9604822, 0.53046876, 0.9579238..."
4,00c19f4fa,"[0.020924012275847437, 0.12606051674692606, 0....",2,"[0.0014462878, 0.08401318, 0.8484335, 0.056610...","[-2.509712, 0.45661384, 0.80324507, 0.8857457,..."


In [59]:
merged_datasets.isnull().mean().mul(100).round(2).rename('% nulos')


PetID                0.00
lgb_pred_score       0.00
AdoptionSpeed        0.00
bert_pred_score      0.10
resnet_pred_score    2.27
Name: % nulos, dtype: float64

In [60]:
# Limpiar nulos (rellenar con arrays de ceros si algún modelo no tiene predicción para un PetID)
merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['resnet_pred_score']]
merged_datasets['bert_pred_score']   = [np.zeros(5) if type(i) is float else i for i in merged_datasets['bert_pred_score']]
merged_datasets['lgb_pred_score']    = [np.zeros(5) if type(i) is float else i for i in merged_datasets['lgb_pred_score']]


In [61]:
all_values = [item for sublist in merged_datasets["resnet_pred_score"] for item in sublist]
min_val = min(all_values)
max_val = max(all_values)

def normalizar_lista(lista):
    return [(x - min_val) / (max_val - min_val) for x in lista]

merged_datasets["resnet_pred_score"] = merged_datasets["resnet_pred_score"].apply(normalizar_lista)

all_values1 = [item for sublist in merged_datasets["bert_pred_score"] for item in sublist]
min_val1 = min(all_values1)
max_val1 = max(all_values1)

def normalizar_lista1(lista):
    return [(x - min_val1) / (max_val1 - min_val1) for x in lista]

merged_datasets["bert_pred_score"] = merged_datasets["bert_pred_score"].apply(normalizar_lista1)

all_values2 = [item for sublist in merged_datasets["lgb_pred_score"] for item in sublist]
min_val2 = min(all_values2)
max_val2 = max(all_values2)

def normalizar_lista2(lista):
    return [(x - min_val2) / (max_val2 - min_val2) for x in lista]

merged_datasets["lgb_pred_score"] = merged_datasets["lgb_pred_score"].apply(normalizar_lista2)


In [62]:
merged_datasets.head()

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score,resnet_pred_score
0,002230dea,"[0.06338951295469104, 0.3074418540495358, 0.43...",1,"[0.009764634848057976, 0.5472034399846415, 0.4...","[0.3858939, 0.60110545, 0.5687438, 0.5231127, ..."
1,0063f83c9,"[0.09184125250627968, 0.16724182833967538, 0.2...",1,"[0.0020327778574799906, 0.005689236591533274, ...","[0.3442009, 0.6307352, 0.5053706, 0.55331373, ..."
2,0073c33d0,"[0.00532296862330662, 0.2323406003115052, 0.23...",3,"[0.0007249530609934081, 0.016793252943771726, ...","[0.33266822, 0.60467386, 0.66474736, 0.5588558..."
3,00bfa5da9,"[0.007909794047651216, 0.09976749683027117, 0....",4,"[8.426123820892333e-05, 0.00040107846939244556...","[0.14036748, 0.3897804, 0.57475185, 0.6277831,..."
4,00c19f4fa,"[0.020499033847363714, 0.13094700650423355, 0....",2,"[0.001448048319537855, 0.08411544279376652, 0....","[0.19757868, 0.56558925, 0.6085932, 0.6188285,..."


In [63]:
# Optimización con Optuna para 3 pesos
def objective(trial):
    # Definir pesos para los tres modelos
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_bert = trial.suggest_float('w_bert', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    
    # Normalización
    total_w = w_lgb + w_bert + w_resnet
    if total_w == 0: return 0
    
    # Cálculo vectorizado para mayor velocidad
    # Convertimos las columnas de scores en una matriz 3D o sumamos directamente
    lgb_scores = np.stack(merged_datasets['lgb_pred_score'].values)
    bert_scores = np.stack(merged_datasets['bert_pred_score'].values)
    resnet_scores = np.stack(merged_datasets['resnet_pred_score'].values)
    
    combined_scores = (
        (w_lgb / total_w) * lgb_scores + 
        (w_bert / total_w) * bert_scores + 
        (w_resnet / total_w) * resnet_scores
    )
    
    preds_final = np.argmax(combined_scores, axis=1)
    
    return cohen_kappa_score(merged_datasets['AdoptionSpeed'], preds_final, weights='quadratic')

In [64]:
# Ejecutar el estudio
STORAGE_URL = "sqlite:///../work/db.sqlite3"
study_blend = optuna.create_study(
    direction='maximize',
    storage=STORAGE_URL,
    study_name="Ensemble_LGB_BERT_ResNet",
    load_if_exists=True
)
study_blend.optimize(objective, n_trials=100)

[I 2026-05-09 00:38:02,791] Using an existing study with name 'Ensemble_LGB_BERT_ResNet' instead of creating a new one.
[I 2026-05-09 00:38:03,085] Trial 1701 finished with value: 0.3532692521828009 and parameters: {'w_lgb': 0.7087659607057254, 'w_bert': 0.1424328376465346, 'w_resnet': 0.17972517078675287}. Best is trial 483 with value: 0.4235531747629919.
[I 2026-05-09 00:38:03,138] Trial 1702 finished with value: 0.3380931256326625 and parameters: {'w_lgb': 0.818626887139089, 'w_bert': 0.05599281166603747, 'w_resnet': 0.22111730482696185}. Best is trial 483 with value: 0.4235531747629919.
[I 2026-05-09 00:38:03,200] Trial 1703 finished with value: 0.3578549926024428 and parameters: {'w_lgb': 0.636151390793837, 'w_bert': 0.1015288497195053, 'w_resnet': 0.2541242604205714}. Best is trial 483 with value: 0.4235531747629919.
[I 2026-05-09 00:38:03,263] Trial 1704 finished with value: 0.3322043546459257 and parameters: {'w_lgb': 0.6865440682830056, 'w_bert': 0.6530439537613856, 'w_resnet'

In [65]:
# Resultados
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

print(f"Mejor Kappa: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")

Mejor Kappa: 0.4236
Pesos óptimos: {'w_lgb': 0.6420695190692808, 'w_bert': 0.12797744543226441, 'w_resnet': 0.19324590533142716}


In [66]:
best_kappa = study_blend.best_trial.value
print(f"Mejor puntuación Kappa: {best_kappa}")

Mejor puntuación Kappa: 0.4235531747629919


In [67]:
# Crear la columna de predicción final optimizada

def _prediction_vector(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    return np.zeros(5, dtype=float)

merged_datasets['blend_pred_score'] = [
    (best_params['w_lgb'] / sum_best_w) * _prediction_vector(r['lgb_pred_score']) +
    (best_params['w_bert'] / sum_best_w) * _prediction_vector(r['bert_pred_score']) +
    (best_params['w_resnet'] / sum_best_w) * _prediction_vector(r['resnet_pred_score'])
    for _, r in merged_datasets.iterrows()
]

In [68]:
merged_datasets[['lgb_pred_score', 'bert_pred_score', 'resnet_pred_score']]

,lgb_pred_score,bert_pred_score,resnet_pred_score
0,"[0.06338951295469104, 0.3074418540495358, 0.43...","[0.009764634848057976, 0.5472034399846415, 0.4...","[0.3858939, 0.60110545, 0.5687438, 0.5231127, ..."
1,"[0.09184125250627968, 0.16724182833967538, 0.2...","[0.0020327778574799906, 0.005689236591533274, ...","[0.3442009, 0.6307352, 0.5053706, 0.55331373, ..."
2,"[0.00532296862330662, 0.2323406003115052, 0.23...","[0.0007249530609934081, 0.016793252943771726, ...","[0.33266822, 0.60467386, 0.66474736, 0.5588558..."
3,"[0.007909794047651216, 0.09976749683027117, 0....","[8.426123820892333e-05, 0.00040107846939244556...","[0.14036748, 0.3897804, 0.57475185, 0.6277831,..."
4,"[0.020499033847363714, 0.13094700650423355, 0....","[0.001448048319537855, 0.08411544279376652, 0....","[0.19757868, 0.56558925, 0.6085932, 0.6188285,..."
...,...,...,...
2994,"[0.06309052792363055, 0.3764763448547162, 0.15...","[0.0029909178783425424, 0.06526598962488048, 0...","[0.2567495, 0.52802324, 0.5217696, 0.56667143,..."
2995,"[0.2851502711379372, 0.18927432061548044, 0.34...","[0.0014949311689542628, 0.22506888091349558, 0...","[0.24999852, 0.5179931, 0.6030275, 0.5335944, ..."
2996,"[0.3074209837077267, 0.27131064076121053, 0.24...","[0.015637692593982188, 0.32118321502428354, 0....","[0.20750931, 0.67806304, 0.6485008, 0.5761223,..."
2997,"[0.03476466291855015, 0.6590189508734386, 0.14...","[0.0007484671007292614, 0.019214953625744838, ...","[0.3344465, 0.51293564, 0.5606445, 0.54551184,..."


In [69]:
#merged_datasets['blend_pred_score']

In [70]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred_score'].apply(np.argmax), 
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))

In [71]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['bert_pred_score'].apply(np.argmax), 
                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['bert_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [72]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['resnet_pred_score'].apply(np.argmax), 
                    title = 'ResNet Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['resnet_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [73]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blend_pred_score'].apply(np.argmax), 
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['blend_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [74]:
fig = plot_param_importances(study_blend, evaluator= FanovaImportanceEvaluator(seed=24))
fig.update_layout(
    title="Importancia de los Modelos en el Ensamble",
    xaxis_title="Importancia relativa",
    yaxis_title="Modelos",
    template="plotly_white"
)
fig.show()



In [75]:
#fig_contour = plot_contour(study_blend, params=['w_lgb', 'w_bert', 'w_resnet'])

#fig_contour.update_layout(
#    title="Interacción de Pesos y Rendimiento (Kappa)",
#    width=900,
#    height=800
#)

#fig_contour.show()

In [76]:
# Guardar el resultado final en un archivo temporal y subirlo a Optuna
final_filename = "merged_predictions_optimized.joblib"
dump(merged_datasets, final_filename)


['merged_predictions_optimized.joblib']

In [77]:
# Subir el archivo como artefacto del mejor trial
#artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)
#upload_artifact(
#    trial=study_blend.best_trial, 
#    file_path=final_filename, 
#    artifact_store=artifact_store
#)
